# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is a **ranking** question ("which pages first?"), not a plain yes/no — the
`training-honest-models` skill maps that shape to "any classifier's probability, evaluated at
precision@K," not to accuracy or a hard 0/1 prediction.

**Starting method: Logistic Regression.** Readable — I can name and sign-check every
coefficient before trusting the ranking it produces, which matters more here than a fraction of
a point of precision. **Second method: Random Forest.** Same features, same split — the toolkit's
own guidance is "readable, then stronger," and a forest can pick up feature interactions
(e.g. position mattering differently at high vs. low volume) that a linear model can't. I only
keep the added complexity if it actually beats logistic regression on the same table below —
otherwise the extra opacity isn't earning its place.

In [6]:
%pip -q install duckdb scikit-learn

import duckdb
import numpy as np
import pandas as pd
from getpass import getpass
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Auth: register the HF read token as a DuckDB secret. Never paste a token into a cell ---
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face READ token (from a Colab Secret named HF_TOKEN ideally): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DAILY_FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"

FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"

# --- Rebuild the exact w03/w04 panel ---
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks)      AS feb_clicks,
           AVG(gsc_avg_position) AS feb_avg_position,
           SUM(ga4_sessions)     AS feb_sessions,
           SUM(sessions_ai)      AS feb_ai_sessions,
           SUM(CASE WHEN report_date < DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h1,
           SUM(CASE WHEN report_date >= DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h2
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS mar_impressions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{LABEL_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = feat.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
panel = panel[panel["feb_impressions"] > 0].copy()
panel["declined_next_month"] = (
    panel["mar_impressions"] < panel["feb_impressions"] * 0.8
).astype(int)

content_meta = con.sql(f"""
    SELECT content_hash_id, content_created_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()
panel = panel.merge(content_meta, on="content_hash_id", how="left")
panel["content_created_date"] = pd.to_datetime(panel["content_created_date"])
as_of = pd.Timestamp(f"{FEATURE_MONTH}-28")
panel["age_days"] = (as_of - panel["content_created_date"]).dt.days
panel["declining_now"] = (panel["feb_h2"] < panel["feb_h1"] * 0.8).astype(int)

# The w04 baseline rule, recomputed identically here for a fair, same-notebook comparison.
panel["stale"] = ((panel["age_days"] >= 90) & (panel["age_days"] < 365)).astype(int)
panel["visible"] = (panel["feb_impressions"] >= 500).astype(int)
panel["baseline_score"] = panel["stale"] * panel["visible"] * panel["feb_impressions"]

print(f"Panel rows: {len(panel):,}  |  Distinct clients: {panel['client_hash_id'].nunique()}")
print(f"Base rate (declined_next_month): {panel['declined_next_month'].mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Panel rows: 145,279  |  Distinct clients: 42
Base rate (declined_next_month): 26.1%


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Time-awareness is already baked into the panel, not the split.** Every row's features
come from February only and the label from March only (validated in w03's leakage trap) — so
there's no chronological leakage left for the train/test split to introduce or fix.

**What the split still has to guard against is client memorization.** `feb_avg_position`,
`feb_ai_sessions`, and similar columns carry a client's own baseline SEO maturity — a random
row-level split would let the model see 80% of a given client's pages in training and be
"tested" on the other 20% of the *same* client, which quietly rewards memorizing that client's
quirks rather than learning a pattern that generalizes to a client the model has never seen.
Per the `hunting-leakage-and-validating` skill, I split by `client_hash_id`
(`GroupShuffleSplit`) so entire clients land in either train or test, never both — and I report
the naive random-row split next to it, because the *gap* between the two is itself the honest
finding about how much memorization was happening.

In [7]:
FEATURE_COLS = [
    "feb_impressions", "feb_clicks", "feb_avg_position",
    "feb_sessions", "feb_ai_sessions", "age_days", "declining_now",
]
LABEL_COL = "declined_next_month"

model_df = panel.dropna(subset=FEATURE_COLS + [LABEL_COL]).copy()
print(f"Rows usable for modeling (after dropping missing age/features): {len(model_df):,} / {len(panel):,}")

# --- Grouped split: whole clients held out, never split across train/test ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["client_hash_id"]))
train_grouped, test_grouped = model_df.iloc[train_idx], model_df.iloc[test_idx]

overlap = set(train_grouped["client_hash_id"]) & set(test_grouped["client_hash_id"])
print(f"Grouped split -- train rows: {len(train_grouped):,}, test rows: {len(test_grouped):,}")
print(f"Train clients: {train_grouped['client_hash_id'].nunique()}, "
      f"test clients: {test_grouped['client_hash_id'].nunique()}, "
      f"overlapping clients (should be 0): {len(overlap)}")

# --- Naive comparison: plain random row split, same 80/20 ratio ---
from sklearn.model_selection import train_test_split
train_random, test_random = train_test_split(
    model_df, test_size=0.2, random_state=RANDOM_SEED
)
random_overlap = set(train_random["client_hash_id"]) & set(test_random["client_hash_id"])
print(f"\nRandom split (for comparison only) -- overlapping clients: {len(random_overlap)} "
      f"(expected to be > 0 -- this is exactly the leakage risk the grouped split avoids)")

Rows usable for modeling (after dropping missing age/features): 72,849 / 145,279
Grouped split -- train rows: 70,840, test rows: 2,009
Train clients: 20, test clients: 5, overlapping clients (should be 0): 0

Random split (for comparison only) -- overlapping clients: 23 (expected to be > 0 -- this is exactly the leakage risk the grouped split avoids)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def evaluate_split(train_df, test_df, label, tag):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURE_COLS])
    X_test = scaler.transform(test_df[FEATURE_COLS])
    y_train = train_df[LABEL_COL].values
    y_test = test_df[LABEL_COL].values

    logreg = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
    logreg.fit(X_train, y_train)
    logreg_scores = logreg.predict_proba(X_test)[:, 1]

    rf = RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    rf_scores = rf.predict_proba(X_test)[:, 1]

    baseline_scores = test_df["baseline_score"].values
    base_rate = y_test.mean()

    rows = []
    for k in (20, 50):
        rows.append({
            "split": tag, "k": k, "base_rate": base_rate,
            "baseline_precision": precision_at_k(baseline_scores, y_test, k),
            "logreg_precision": precision_at_k(logreg_scores, y_test, k),
            "rf_precision": precision_at_k(rf_scores, y_test, k),
        })
    return pd.DataFrame(rows), logreg, rf, X_test, y_test, logreg_scores, rf_scores


results_grouped, logreg_g, rf_g, X_test_g, y_test_g, logreg_scores_g, rf_scores_g = evaluate_split(
    train_grouped, test_grouped, LABEL_COL, "grouped_by_client"
)
results_random, logreg_r, rf_r, X_test_r, y_test_r, logreg_scores_r, rf_scores_r = evaluate_split(
    train_random, test_random, LABEL_COL, "random_row_split"
)

comparison = pd.concat([results_grouped, results_random], ignore_index=True)
# Format as percentages for THIS print only -- never touch pd.set_option globally,
# it silently corrupts every later float print in the notebook (feb_impressions,
# coefficients, etc. would get multiplied by 100 and stamped with a % they don't mean).
pct_cols = ["base_rate", "baseline_precision", "logreg_precision", "rf_precision"]
display_table = comparison.copy()
for col in pct_cols:
    display_table[col] = display_table[col].map(lambda x: f"{x:.1%}")
print(display_table.to_string(index=False))
print("\n-> The gap between 'grouped_by_client' and 'random_row_split' rows at the same k")
print("   IS the memorization check from hunting-leakage-and-validating -- a big gap would mean")
print("   the random-split number was flattering the model with client-specific memorization.")

            split  k base_rate baseline_precision logreg_precision rf_precision
grouped_by_client 20     40.0%              20.0%            85.0%        70.0%
grouped_by_client 50     40.0%              20.0%            78.0%        70.0%
 random_row_split 20     30.5%              25.0%            40.0%       100.0%
 random_row_split 50     30.5%              20.0%            50.0%        98.0%

-> The gap between 'grouped_by_client' and 'random_row_split' rows at the same k
   IS the memorization check from hunting-leakage-and-validating -- a big gap would mean
   the random-split number was flattering the model with client-specific memorization.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
print("== Feature importances ==")
print("\nLogistic Regression coefficients (standardized -- sign and magnitude both matter):")
coef_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "logreg_coef": logreg_g.coef_[0],
}).sort_values("logreg_coef", key=abs, ascending=False)
print(coef_table.to_string(index=False))

print("\nRandom Forest feature importances:")
rf_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "rf_importance": rf_g.feature_importances_,
}).sort_values("rf_importance", ascending=False)
print(rf_importance.to_string(index=False))

print("\n== Where is the (grouped-split) model most wrong? ==")
test_eval = test_grouped.copy()
test_eval["rf_score"] = rf_scores_g
test_eval["rf_pred_top20"] = test_eval["rf_score"] >= test_eval["rf_score"].nlargest(20).min()

vol_bins = [-1, 100, 500, 2000, 10000, 10_000_000]
vol_labels = ["<100", "100-500", "500-2000", "2000-10000", "10000+"]
test_eval["volume_tier"] = pd.cut(test_eval["feb_impressions"], bins=vol_bins, labels=vol_labels)
error_by_tier = test_eval.groupby("volume_tier", observed=True).agg(
    n=("declined_next_month", "size"),
    actual_decline_rate=("declined_next_month", "mean"),
    avg_predicted_score=("rf_score", "mean"),
).reset_index()
print(error_by_tier.to_string(index=False))

print("\n== 3 concrete wrong cases (highest-confidence false positives) ==")
false_positives = test_eval[test_eval["declined_next_month"] == 0].nlargest(3, "rf_score")
print(false_positives[
    ["content_hash_id", "feb_impressions", "age_days", "declining_now", "rf_score", "declined_next_month"]
].to_string(index=False))

print("\n== 3 concrete wrong cases (highest-confidence false negatives) ==")
false_negatives = test_eval[test_eval["declined_next_month"] == 1].nsmallest(3, "rf_score")
print(false_negatives[
    ["content_hash_id", "feb_impressions", "age_days", "declining_now", "rf_score", "declined_next_month"]
].to_string(index=False))

== Feature importances ==

Logistic Regression coefficients (standardized -- sign and magnitude both matter):
         feature  logreg_coef
   declining_now     0.426426
 feb_impressions    -0.345285
feb_avg_position    -0.188349
        age_days    -0.180243
 feb_ai_sessions    -0.074630
    feb_sessions    -0.053638
      feb_clicks     0.013452

Random Forest feature importances:
         feature  rf_importance
 feb_impressions       0.373655
   declining_now       0.200903
        age_days       0.173610
      feb_clicks       0.121794
feb_avg_position       0.070040
    feb_sessions       0.058147
 feb_ai_sessions       0.001851

== Where is the (grouped-split) model most wrong? ==
volume_tier    n  actual_decline_rate  avg_predicted_score
       <100 1728             0.414352             0.410945
    100-500  180             0.350000             0.232353
   500-2000   72             0.277778             0.177353
 2000-10000   27             0.148148             0.147697
     1000

In [10]:
import json
from pathlib import Path

metrics = {
    "feature_month": FEATURE_MONTH,
    "label_month": LABEL_MONTH,
    "random_seed": RANDOM_SEED,
    "feature_cols": FEATURE_COLS,
    "comparison_table": comparison.to_dict(orient="records"),
    "logreg_coefficients": coef_table.to_dict(orient="records"),
    "rf_importances": rf_importance.to_dict(orient="records"),
    "grouped_split_train_clients": int(train_grouped["client_hash_id"].nunique()),
    "grouped_split_test_clients": int(test_grouped["client_hash_id"].nunique()),
}
metrics_path = Path("work/outputs/model_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Wrote {metrics_path} -- commit this one, same as w04's baseline_metrics.json.")

Wrote work/outputs/model_metrics.json -- commit this one, same as w04's baseline_metrics.json.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.